# 03 — Frozen evaluation, monitoring and export
Inputs: frozen `runs/final` bundle, fixed private dataset. Cap: 10 minutes.
Output: once-only held-out scorecard, SQLite replay, ZIP and completion checksums.
The final evaluation refuses a second start. If already completed, read the saved evidence.
A failed/interrupted evaluation requires investigation; never delete its start marker to hide reuse.
The controlled update is deferred: support groups are unreviewed. No automatic retraining.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
# Local: run from repo root or training/notebooks. Colab: copy private project to /content/svo.
ROOT = Path('/content/svo') if Path('/content/svo/training').exists() else Path.cwd()
while not (ROOT / 'training').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'training/src/svo').exists(), 'Set ROOT to the extracted project root'
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'training/src')
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
PYTHON = str(ROOT / 'training/.venv/bin/python') if (ROOT / 'training/.venv/bin/python').exists() else sys.executable
DATA = ROOT / 'training/data/processed/messages.jsonl'
DRIVE_ROOT = Path('/content/drive/MyDrive/ScamVariantObservatory')
def run(*args):
    subprocess.run([PYTHON, *map(str,args)], check=True, cwd=ROOT, env=os.environ.copy())
run('-c', 'import numpy, sklearn, joblib; print("Core dependencies ready")')


In [ ]:
RUN = ROOT / 'training/runs/final'
assert (RUN / 'freeze.json').exists(), 'Complete development selection first'
run('-m','unittest','discover','-s','training/tests','-v')
run('training/scripts/verify_bundle.py', RUN / 'bundle')


## Final report boundary
Execute only after selection is frozen and sufficient time remains. These results must not tune the model.


In [ ]:
if not (RUN / 'FINAL_EVALUATION_COMPLETE.json').exists():
    run('-m','svo.evaluate','--data',DATA,'--run',RUN,'--simulation')
print(json.loads((RUN / 'scorecard.json').read_text()))
assert (RUN / 'FINAL_EVALUATION_COMPLETE.json').exists()


In [ ]:
# Optional Colab checkpoint. Mount Drive before setting this flag.
COPY_TO_DRIVE = False
if COPY_TO_DRIVE:
    import shutil
    assert DRIVE_ROOT.parent.exists(), 'Mount Drive first'
    destination = DRIVE_ROOT / 'training-checkpoints' / RUN.name
    shutil.copytree(RUN, destination, dirs_exist_ok=False)
    print('Checkpoint copied; treat it as private.')
print('Local stage artifacts:', RUN)
